# Experiment: IQC Stage 1 Starter Research Slate

Objective:
- Build a first-wave slate of alpha families for IQC Stage 1.
- Keep the work reproducible, low-duplication, and easy to extend into BRAIN simulations.


In [1]:
# Setup: lightweight structures for notebook review and ledger logging
from __future__ import annotations

from dataclasses import asdict, dataclass
from datetime import date
from pathlib import Path
import csv

LEDGER_PATH = Path('../../research/alpha_ledger.csv')
LEDGER_FIELDS = [
    'date',
    'family_name',
    'thesis_summary',
    'data_class',
    'formula_summary',
    'region',
    'universe',
    'delay',
    'neutralization',
    'truncation',
    'decay',
    'status',
    'correlation_risk',
    'next_action',
    'notes',
]
VALID_STATUSES = {'planned', 'screened', 'simulated', 'submittable', 'submitted', 'retired'}

@dataclass
class AlphaFamily:
    family_name: str
    thesis_summary: str
    data_class: str
    economic_intuition: str
    candidate_pattern: str
    settings_sweep: str
    acceptance_rule: str
    stop_rule: str
    region: str = 'USA'
    universe: str = 'TOP3000'
    delay: str = '1'
    neutralization: str = 'Subindustry'
    truncation: str = '0.08'
    decay: str = '4'

    def preview(self) -> dict[str, str]:
        return {
            'family_name': self.family_name,
            'data_class': self.data_class,
            'candidate_pattern': self.candidate_pattern,
            'acceptance_rule': self.acceptance_rule,
            'stop_rule': self.stop_rule,
        }

def make_ledger_row(
    family: AlphaFamily,
    formula_summary: str,
    status: str = 'planned',
    correlation_risk: str = 'unknown',
    next_action: str = 'Define first simulation-ready expression.',
    notes: str = '',
) -> dict[str, str]:
    if status not in VALID_STATUSES:
        raise ValueError(f'Unsupported status: {status}')
    row = {
        'date': date.today().isoformat(),
        'family_name': family.family_name,
        'thesis_summary': family.thesis_summary,
        'data_class': family.data_class,
        'formula_summary': formula_summary,
        'region': family.region,
        'universe': family.universe,
        'delay': family.delay,
        'neutralization': family.neutralization,
        'truncation': family.truncation,
        'decay': family.decay,
        'status': status,
        'correlation_risk': correlation_risk,
        'next_action': next_action,
        'notes': notes,
    }
    return {field: row[field] for field in LEDGER_FIELDS}

def append_ledger_row(row: dict[str, str]) -> None:
    with LEDGER_PATH.open('a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=LEDGER_FIELDS)
        writer.writerow(row)

LEDGER_PATH


WindowsPath('../../research/alpha_ledger.csv')

## Research rules

- Start with price-volume families first.
- Do not simulate a family until it has a clear thesis, operator pattern, settings sweep, and stop rule.
- Log every concrete tested variant using one of the fixed statuses: `planned`, `screened`, `simulated`, `submittable`, `submitted`, `retired`.


In [2]:
# Starter alpha slate: first-wave families to evaluate
starter_families = [
    AlphaFamily(
        family_name='pv_reversal_outsized_move',
        thesis_summary='Short-term price overshoots partially mean-revert after unusually large moves.',
        data_class='price_volume',
        economic_intuition='Very large moves often attract temporary liquidity imbalances that reverse once order pressure fades.',
        candidate_pattern='rank(-ts_delta(close, 1)) with short lookback normalization.',
        settings_sweep='lookback 1 to 5, decay 2 to 6, truncation 0.05 to 0.10.',
        acceptance_rule='Keep only if the first variants are submittable without requiring extreme parameter tuning.',
        stop_rule='Stop after repeated weak variants if only parameter changes remain.',
    ),
    AlphaFamily(
        family_name='pv_reversal_abnormal_volume',
        thesis_summary='Reversal signals are stronger when the initiating move occurs on unusual volume.',
        data_class='price_volume',
        economic_intuition='Volume spikes can mark temporary crowding or exhaustion, which may improve reversal quality.',
        candidate_pattern='Combine negative short-term return with a rank of relative volume spike.',
        settings_sweep='volume lookback 5 to 20, price lookback 1 to 3, decay 2 to 5.',
        acceptance_rule='Keep only if volume conditioning improves distinctness or quality over plain reversal.',
        stop_rule='Stop if it behaves like the base reversal family with no clear incremental value.',
    ),
    AlphaFamily(
        family_name='pv_momentum_liquidity_check',
        thesis_summary='Momentum works better when liquidity is adequate and the signal is not driven by microcap noise.',
        data_class='price_volume',
        economic_intuition='Persistent moves can continue, but only where trading conditions make the signal more reliable.',
        candidate_pattern='rank(ts_delta(close, 5)) filtered or scaled by liquidity proxy.',
        settings_sweep='momentum lookback 5 to 20, liquidity lookback 10 to 30, decay 3 to 8.',
        acceptance_rule='Keep only if the liquidity filter changes the behavior materially, not cosmetically.',
        stop_rule='Stop if it turns into a generic momentum family with only minor filter effects.',
    ),
    AlphaFamily(
        family_name='pv_vol_normalized_dislocation',
        thesis_summary='Price dislocations scaled by recent volatility may identify cleaner reversion opportunities.',
        data_class='price_volume',
        economic_intuition='A move that looks large in raw terms may be normal in a high-volatility name, so normalization can sharpen the signal.',
        candidate_pattern='rank(-ts_delta(close, 3) / ts_std_dev(close, 20)).',
        settings_sweep='return lookback 2 to 5, volatility lookback 10 to 30, decay 2 to 6.',
        acceptance_rule='Keep only if volatility scaling reduces noise or correlation versus other reversal families.',
        stop_rule='Stop if it duplicates the same names and behavior as the simple reversal family.',
    ),
    AlphaFamily(
        family_name='fm_valuation_gap_reversion',
        thesis_summary='Names that look cheap relative to peers may mean-revert as valuation gaps close.',
        data_class='fundamental_model',
        economic_intuition='Cross-sectional valuation extremes can normalize over time, especially after neutralization.',
        candidate_pattern='rank(-valuation metric) with group neutralization.',
        settings_sweep='test one or two valuation metrics, neutralization levels, and modest decay values.',
        acceptance_rule='Keep only if the family stays interpretable and does not rely on obscure combinations.',
        stop_rule='Stop if multiple valuation metrics tell the same story without improving results.',
    ),
    AlphaFamily(
        family_name='fm_quality_profitability',
        thesis_summary='Higher-quality and more profitable firms may outperform weaker peers in the cross section.',
        data_class='fundamental_model',
        economic_intuition='Profitability and quality often capture durable business strength that may not be fully priced.',
        candidate_pattern='rank(quality metric) or combine profitability with leverage discipline.',
        settings_sweep='test one quality metric at a time, then simple two-metric combinations.',
        acceptance_rule='Keep only if single-metric variants already show some promise before combining factors.',
        stop_rule='Stop if combinations become story-fitting rather than economically clean.',
    ),
    AlphaFamily(
        family_name='fm_estimate_change_reaction',
        thesis_summary='Changes in analyst or model expectations may contain delayed cross-sectional information.',
        data_class='fundamental_model',
        economic_intuition='Revisions can move slower through the market than raw price data, creating exploitable lagged effects.',
        candidate_pattern='rank(change in estimate-related field) with conservative smoothing.',
        settings_sweep='change windows 1 to 4 periods, decay 2 to 6, neutralization at sector or subindustry.',
        acceptance_rule='Keep only if the family remains interpretable and distinct from pure momentum.',
        stop_rule='Stop if the signal is just price trend in disguise.',
    ),
    AlphaFamily(
        family_name='adv_relationship_placeholder',
        thesis_summary='Reserve one advanced family for later once the base pipeline is productive.',
        data_class='advanced_placeholder',
        economic_intuition='Advanced datasets may help later, but only after baseline families are already producing learnings.',
        candidate_pattern='Do not implement yet. Pick one advanced dataset only after the base slate is productive.',
        settings_sweep='None for now.',
        acceptance_rule='Open this branch only after price-volume and fundamental/model work is producing viable output.',
        stop_rule='If the base pipeline is still weak, keep this branch closed.',
    ),
]

len(starter_families)


8

## Preview and logging

- Use the preview table to review families before simulation.
- Build a ledger row only after you can state the exact BRAIN expression or tested variant.
- Append rows only after checking that the status and notes are accurate.


In [3]:
# Preview the starter families and create an example ledger row
family_preview = [family.preview() for family in starter_families]
example_row = make_ledger_row(
    starter_families[0],
    formula_summary='rank(-ts_delta(close, 1))',
    status='screened',
    next_action='Simulate the plain reversal baseline before adding filters.',
    notes='Baseline family for learning operator behavior and submission mechanics.',
)

family_preview[:3], example_row

# Uncomment to append the example row after manual review.
# append_ledger_row(example_row)


([{'family_name': 'pv_reversal_outsized_move',
   'data_class': 'price_volume',
   'candidate_pattern': 'rank(-ts_delta(close, 1)) with short lookback normalization.',
   'acceptance_rule': 'Keep only if the first variants are submittable without requiring extreme parameter tuning.',
   'stop_rule': 'Stop after repeated weak variants if only parameter changes remain.'},
  {'family_name': 'pv_reversal_abnormal_volume',
   'data_class': 'price_volume',
   'candidate_pattern': 'Combine negative short-term return with a rank of relative volume spike.',
   'acceptance_rule': 'Keep only if volume conditioning improves distinctness or quality over plain reversal.',
   'stop_rule': 'Stop if it behaves like the base reversal family with no clear incremental value.'},
  {'family_name': 'pv_momentum_liquidity_check',
   'data_class': 'price_volume',
   'candidate_pattern': 'rank(ts_delta(close, 5)) filtered or scaled by liquidity proxy.',
   'acceptance_rule': 'Keep only if the liquidity filter c

## First real step: one family, three variants

Start with `pv_reversal_outsized_move` only. The goal is not to be clever yet. The goal is to learn how small changes affect behavior.

The three variants below are intentionally simple:
- baseline reversal
- reversal normalized by recent volatility
- reversal conditioned on relative volume

Treat these as the first BRAIN-style expressions to try. If your BRAIN editor expects a slightly different operator spelling or field name, adjust there before simulation.


In [4]:
# Three concrete starter variants for the first family
first_family = starter_families[0]

first_family_variants = [
    {
        'variant_name': 'baseline_reversal_1d',
        'formula_summary': 'rank(-ts_delta(close, 1))',
        'why': 'Bet against yesterday\'s outsized move and see the clean baseline behavior first.',
    },
    {
        'variant_name': 'vol_normalized_reversal',
        'formula_summary': 'rank(-(ts_delta(close, 1) / ts_std_dev(close, 20)))',
        'why': 'Scale the move by recent volatility so noisy high-volatility names do not dominate.',
    },
    {
        'variant_name': 'volume_conditioned_reversal',
        'formula_summary': 'rank(-ts_delta(close, 1)) * rank(volume / ts_mean(volume, 20))',
        'why': 'Give more weight to reversals that happen on unusual volume.',
    },
]

first_family_variants


[{'variant_name': 'baseline_reversal_1d',
  'formula_summary': 'rank(-ts_delta(close, 1))',
  'why': "Bet against yesterday's outsized move and see the clean baseline behavior first."},
 {'variant_name': 'vol_normalized_reversal',
  'formula_summary': 'rank(-(ts_delta(close, 1) / ts_std_dev(close, 20)))',
  'why': 'Scale the move by recent volatility so noisy high-volatility names do not dominate.'},
 {'variant_name': 'volume_conditioned_reversal',
  'formula_summary': 'rank(-ts_delta(close, 1)) * rank(volume / ts_mean(volume, 20))',
  'why': 'Give more weight to reversals that happen on unusual volume.'}]

## How to use these variants

1. Start with the baseline variant first.
2. Only after that, try the volatility-normalized version.
3. Try the volume-conditioned version third.
4. Log each one separately in the ledger.
5. Write one sentence on what changed and whether the variant looked better, worse, or just different.


In [5]:
# Preview ledger rows for the three starter variants
variant_rows = [
    make_ledger_row(
        first_family,
        formula_summary=variant['formula_summary'],
        status='screened',
        next_action='Simulate this specific starter variant in BRAIN.',
        notes=variant['variant_name'] + ': ' + variant['why'],
    )
    for variant in first_family_variants
]

variant_rows

# Uncomment one line at a time after review.
# append_ledger_row(variant_rows[0])
# append_ledger_row(variant_rows[1])
# append_ledger_row(variant_rows[2])


[{'date': '2026-04-16',
  'family_name': 'pv_reversal_outsized_move',
  'thesis_summary': 'Short-term price overshoots partially mean-revert after unusually large moves.',
  'data_class': 'price_volume',
  'formula_summary': 'rank(-ts_delta(close, 1))',
  'region': 'USA',
  'universe': 'TOP3000',
  'delay': '1',
  'neutralization': 'Subindustry',
  'truncation': '0.08',
  'decay': '4',
  'status': 'screened',
  'correlation_risk': 'unknown',
  'next_action': 'Simulate this specific starter variant in BRAIN.',
  'notes': "baseline_reversal_1d: Bet against yesterday's outsized move and see the clean baseline behavior first."},
 {'date': '2026-04-16',
  'family_name': 'pv_reversal_outsized_move',
  'thesis_summary': 'Short-term price overshoots partially mean-revert after unusually large moves.',
  'data_class': 'price_volume',
  'formula_summary': 'rank(-(ts_delta(close, 1) / ts_std_dev(close, 20)))',
  'region': 'USA',
  'universe': 'TOP3000',
  'delay': '1',
  'neutralization': 'Subind